<a href="https://colab.research.google.com/github/RoihansLab/Data-Mining-Project/blob/main/youtube_comment_sentiment/Classifcation/Classifcation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-optimize

import pandas as pd
import numpy as np
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from imblearn.over_sampling import SMOTE
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import time
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import time

# Kunci Random State agar hasil selalu sama setiap di-run
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Cell 1 Selesai: Semua library berhasil dimuat dan Random State dikunci!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.1 MB/s eta 0:00:00
✅ Cell 1 Selesai: Semua library berhasil dimuat dan Random State dikunci!


In [ ]:
# Baca dataset hasil Qwen
df = pd.read_csv("/content/hasil_dari_LLM_qwen.csv")
df = df.dropna(subset=['comment'])

# Encode kolom prediksi_ai menjadi angka 0, 1, 2
le = LabelEncoder()
df['target'] = le.fit_transform(df['prediksi_ai'])

mapping_label = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Mapping Label: {mapping_label}")

# Tampilkan 5 data teratas untuk memastikan
display(df[['video_title', 'comment', 'prediksi_ai', 'target']].head())

Mapping Label: {'negatif': np.int64(0), 'netral': np.int64(1), 'positif': np.int64(2)}


,video_title,comment,prediksi_ai,target
0,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Bahas imigrant maroko yg nyebrang ke spanyol d...,netral,1
1,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Menurut saya jika semua pengusaha berpikiran s...,negatif,0
2,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,jujur ya sebagai seorang yang bekerja di bidan...,negatif,0
3,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Gw sebagai illustrator susah bener dapet klien...,negatif,0
4,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Niat untung -> efesiensi phk->produksi meningk...,negatif,0


In [ ]:
# Gabungkan judul dan komen
df['teks_gabungan'] = df['video_title'].fillna('') + " " + df['comment'].fillna('')

# Fungsi untuk huruf kecil dan buang simbol
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['clean_text'] = df['teks_gabungan'].apply(clean_text)

print("✅ Teks berhasil dibersihkan! Ini perbandingan sebelum vs sesudah:")
display(df[['teks_gabungan', 'clean_text']].head())

✅ Teks berhasil dibersihkan! Ini perbandingan sebelum vs sesudah:


,teks_gabungan,clean_text
0,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...
1,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...
2,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...
3,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...
4,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,pekerja latih robot untuk gantikan mereka kiam...


In [ ]:
X_raw = df['clean_text']
y = df['target']

# Split 80% Train, 20% Test
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw) # Ingat: test hanya di-transform

print(f"Jumlah Data Latih (Train): {X_train_tfidf.shape[0]} baris")
print(f"Jumlah Data Uji (Test): {X_test_tfidf.shape[0]} baris")
print(f"Jumlah Fitur TF-IDF (Kosakata unik): {X_train_tfidf.shape[1]} kolom")

Jumlah Data Latih (Train): 3246 baris
Jumlah Data Uji (Test): 812 baris
Jumlah Fitur TF-IDF (Kosakata unik): 1500 kolom


In [ ]:
print("📉 Keseimbangan Kelas Data Latih SEBELUM SMOTE:")
print(y_train.value_counts().sort_index())

# Terapkan SMOTE
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_tfidf, y_train)

print("\n📈 Keseimbangan Kelas Data Latih SETELAH SMOTE:")
print(y_train_smote.value_counts().sort_index())

📉 Keseimbangan Kelas Data Latih SEBELUM SMOTE:
target
0     627
1    2523
2      96
Name: count, dtype: int64

📈 Keseimbangan Kelas Data Latih SETELAH SMOTE:
target
0    2523
1    2523
2    2523
Name: count, dtype: int64


In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    "Naive Bayes": MultinomialNB(), # Naive Bayes tidak punya parameter class_weight
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
}

base_results = {}
print("=== HASIL EVALUASI MODEL LENGKAP (SEBELUM TUNING) ===")

for name, model in models.items():
    # Latih model
    model.fit(X_train_smote, y_train_smote)

    # Prediksi
    y_pred = model.predict(X_test_tfidf)

    print(f"\n======================================================")
    print(f"📊 REPORT: {name.upper()}")
    print("======================================================")
    # Tampilkan Precision, Recall, dan F1-Score per kelas
    print(classification_report(y_test, y_pred, target_names=['Negatif', 'Netral', 'Positif'], digits=4))

    # Simpan F1-Macro untuk perbandingan di grafik nanti
    f1_mac = f1_score(y_test, y_pred, average='macro')
    base_results[name] = {'F1-Macro': f1_mac}

=== HASIL EVALUASI MODEL LENGKAP (SEBELUM TUNING) ===

📊 REPORT: DECISION TREE
              precision    recall  f1-score   support

     Negatif     0.3929    0.4204    0.4062       157
      Netral     0.8488    0.8098    0.8289       631
     Positif     0.0952    0.1667    0.1212        24

    accuracy                         0.7155       812
   macro avg     0.4456    0.4656    0.4521       812
weighted avg     0.7384    0.7155    0.7262       812


📊 REPORT: NAIVE BAYES
              precision    recall  f1-score   support

     Negatif     0.3860    0.6688    0.4895       157
      Netral     0.9045    0.6006    0.7219       631
     Positif     0.0413    0.2083    0.0690        24

    accuracy                         0.6022       812
   macro avg     0.4440    0.4926    0.4268       812
weighted avg     0.7788    0.6022    0.6577       812


📊 REPORT: RANDOM FOREST
              precision    recall  f1-score   support

     Negatif     0.5217    0.4586    0.4881       157
  

In [ ]:
search_spaces = {
    "Decision Tree": {'max_depth': Integer(5, 50), 'min_samples_split': Integer(2, 20)},
    "Naive Bayes": {'alpha': Real(0.01, 10.0, prior='log-uniform')},
    "Random Forest": {'n_estimators': Integer(50, 200), 'max_depth': Integer(5, 50)},
    "Logistic Regression": {'C': Real(0.01, 100.0, prior='log-uniform'), 'solver': Categorical(['lbfgs', 'liblinear'])}
}

tuned_results = {}
best_models = {}

print("⏳ Sedang memproses Bayesian Hyperparameter Tuning...")

for name, model in models.items():
    start_time = time.time()

    # Tuning Model
    bayes_search = BayesSearchCV(
        estimator=model, search_spaces=search_spaces[name],
        n_iter=15, cv=3, scoring='f1_macro', n_jobs=-1, random_state=RANDOM_STATE
    )
    bayes_search.fit(X_train_smote, y_train_smote)

    # Prediksi menggunakan model terbaik
    best_model = bayes_search.best_estimator_
    best_models[name] = best_model
    y_pred = best_model.predict(X_test_tfidf)

    end_time = time.time()

    print(f"\n======================================================")
    print(f"🎯 REPORT SETELAH TUNING: {name.upper()}")
    print(f"⏱️ Waktu Tuning: {(end_time - start_time):.2f} detik")
    print(f"🔧 Parameter Terbaik: {bayes_search.best_params_}")
    print("======================================================")
    # Tampilkan Precision, Recall, dan F1-Score per kelas
    print(classification_report(y_test, y_pred, target_names=['Negatif', 'Netral', 'Positif'], digits=4))

    # Simpan hasil untuk grafik
    f1_mac = f1_score(y_test, y_pred, average='macro')
    tuned_results[name] = {'F1-Macro': f1_mac, 'Best_Params': bayes_search.best_params_}

⏳ Sedang memproses Bayesian Hyperparameter Tuning...

🎯 REPORT SETELAH TUNING: DECISION TREE
⏱️ Waktu Tuning: 51.03 detik
🔧 Parameter Terbaik: OrderedDict({'max_depth': 46, 'min_samples_split': 2})
              precision    recall  f1-score   support

     Negatif     0.4211    0.4076    0.4142       157
      Netral     0.8454    0.8320    0.8387       631
     Positif     0.1026    0.1667    0.1270        24

    accuracy                         0.7303       812
   macro avg     0.4563    0.4688    0.4600       812
weighted avg     0.7414    0.7303    0.7356       812


🎯 REPORT SETELAH TUNING: NAIVE BAYES
⏱️ Waktu Tuning: 4.48 detik
🔧 Parameter Terbaik: OrderedDict({'alpha': 0.010253943538922507})
              precision    recall  f1-score   support

     Negatif     0.3707    0.6115    0.4615       157
      Netral     0.8686    0.6498    0.7434       631
     Positif     0.0247    0.0833    0.0381        24

    accuracy                         0.6256       812
   macro avg     

Kode baru

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# Algoritma Modeling
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Metriks Evaluasi
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

In [3]:
df = pd.read_csv("/content/hasil_dari_LLM_qwen.csv")
df

,video_title,comment,likes,bahasa,prediksi_ai
0,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Bahas imigrant maroko yg nyebrang ke spanyol d...,413,Indonesia,netral
1,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Menurut saya jika semua pengusaha berpikiran s...,189,Indonesia,negatif
2,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,jujur ya sebagai seorang yang bekerja di bidan...,6,Indonesia,negatif
3,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Gw sebagai illustrator susah bener dapet klien...,588,Indonesia,negatif
4,Pekerja Latih Robot Untuk Gantikan Mereka! KIA...,Niat untung -> efesiensi phk->produksi meningk...,85,Indonesia,negatif
...,...,...,...,...,...
4053,AI is taking their jobs. But they're fighting ...,There’s a real awakening that cannot be stoppe...,0,Inggris,netral
4054,AI is taking their jobs. But they're fighting ...,I'm a software engineer with 20 years of exper...,0,Inggris,netral
4055,AI is taking their jobs. But they're fighting ...,so many way to discard that... I'll just take ...,2,Inggris,netral
4056,AI is taking their jobs. But they're fighting ...,Are you a software engineer? Because if you we...,0,Inggris,netral


In [8]:
df['text_input'] = "Judul: " + df['video_title'].fillna('') + " | Komentar: " + df['comment'].fillna('')

# 2. Pisahkan Fitur Teks (X_text) dan Target Label (y_raw)
X_text = df['text_input']
y_raw = df['prediksi_ai']

# Tampilkan 5 data pertama untuk memastikan bentuk teksnya sudah sesuai
df[['text_input', 'prediksi_ai']].head()

,text_input,prediksi_ai
0,Judul: Pekerja Latih Robot Untuk Gantikan Mere...,netral
1,Judul: Pekerja Latih Robot Untuk Gantikan Mere...,negatif
2,Judul: Pekerja Latih Robot Untuk Gantikan Mere...,negatif
3,Judul: Pekerja Latih Robot Untuk Gantikan Mere...,negatif
4,Judul: Pekerja Latih Robot Untuk Gantikan Mere...,negatif


In [17]:
le = LabelEncoder()
y = le.fit_transform(y_raw)

# Tampilkan label asli dan hasil encoding secara berdampingan
pd.DataFrame({
    'label_asli (y_raw)': y_raw,
    'label_encoded (y)': y
}).head(10)

,label_asli (y_raw),label_encoded (y)
0,netral,1
1,negatif,0
2,negatif,0
3,negatif,0
4,negatif,0
5,netral,1
6,negatif,0
7,negatif,0
8,netral,1
9,netral,1


In [18]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = tfidf.fit_transform(X_text)

# Ubah sparse matrix menjadi DataFrame Pandas
df_tfidf = pd.DataFrame(
    X.toarray(),
    columns=tfidf.get_feature_names_out()
)

# Tampilkan 10 data pertama
df_tfidf.head(10)

,000,10,10 000,10 tahun,10 years,100,100 years,11,12,13,...,your own,youre,yourself,youth,youth unemployment,youtube,youtuber,zaman,zaman dulu,zero
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.076452,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
from sklearn.decomposition import TruncatedSVD

# TruncatedSVD aman untuk Sparse Matrix dan cocok untuk Teks
svd = TruncatedSVD(n_components=300, random_state=42)
X_reduced = svd.fit_transform(X)

print(f"Bentuk Matriks Setelah Diompres: {X_reduced.shape}")
# Catatan: Jika pakai ini, untuk Naïve Bayes gunakan GaussianNB(), bukan MultinomialNB().

Bentuk Matriks Setelah Diompres: (4058, 300)


In [31]:
# 1. Buat nama kolom untuk 300 komponen hasil SVD (SVD_1, SVD_2, ..., SVD_300)
svd_columns = [f"SVD_{i+1}" for i in range(300)]

# 2. Ubah array X_reduced menjadi DataFrame Pandas
df_svd = pd.DataFrame(X_reduced, columns=svd_columns)

# 3. Tampilkan 10 baris pertama
df_svd.head(10)

,SVD_1,SVD_2,SVD_3,SVD_4,SVD_5,SVD_6,SVD_7,SVD_8,SVD_9,SVD_10,...,SVD_291,SVD_292,SVD_293,SVD_294,SVD_295,SVD_296,SVD_297,SVD_298,SVD_299,SVD_300
0,0.168355,0.467059,-0.182985,-0.467645,-0.005346,-0.002333,0.005575,-0.102215,0.000524,0.003006,...,-0.016906,-0.033045,-0.025490,0.015145,0.018031,-0.004813,0.027746,0.010661,-0.010783,0.014693
1,0.089212,0.250574,-0.073830,-0.136872,-0.000618,-0.002087,0.001478,0.199790,-0.000429,0.001302,...,-0.004285,0.033394,-0.041829,-0.010071,0.009212,0.030568,0.020376,0.005802,0.032941,0.015178
2,0.080810,0.213350,-0.057516,-0.123228,0.003517,-0.000933,-0.001533,0.224694,0.004588,-0.001078,...,-0.017610,-0.017123,-0.000154,-0.039459,0.017027,-0.014552,-0.008116,-0.051775,0.014907,-0.016448
3,0.079439,0.212857,-0.061419,-0.110379,0.004451,-0.004407,0.004828,0.226552,0.009068,-0.004342,...,-0.044380,-0.011999,-0.024871,-0.008886,0.013951,-0.046129,0.042114,0.001029,0.035624,0.000897
4,0.116229,0.321881,-0.124721,-0.315724,-0.003648,-0.001463,0.002880,-0.014881,0.000299,0.000075,...,-0.010608,0.020576,0.018105,-0.004168,-0.020167,-0.014700,0.016215,0.004819,-0.004126,-0.011354
5,0.099199,0.276513,-0.102771,-0.247091,-0.001200,-0.002913,0.006027,0.042735,-0.000124,0.002120,...,-0.017097,-0.029854,0.028280,0.027437,-0.011139,0.005814,-0.011233,-0.018057,-0.030351,0.006176
6,0.139070,0.373214,-0.133758,-0.354980,-0.001430,0.000830,-0.002893,-0.032937,0.000096,-0.002749,...,0.001594,-0.007395,-0.001265,0.036184,0.018548,0.019116,-0.010534,0.006399,-0.015822,-0.011243
7,0.078667,0.225206,-0.067143,-0.108798,0.000070,-0.002622,0.005246,0.253511,0.001700,0.001139,...,-0.053313,0.010639,0.024966,-0.055299,-0.043497,-0.003570,0.017641,0.007076,-0.006433,0.033648
8,0.156249,0.379261,-0.141587,-0.403708,0.032831,-0.044728,-0.011350,-0.118075,0.009669,-0.023186,...,0.004687,-0.049840,0.031890,-0.037405,0.017011,0.026715,0.007421,0.018117,0.020950,-0.013717
9,0.128097,0.356405,-0.136157,-0.334980,-0.004621,-0.002276,0.005483,0.009606,0.000738,0.007969,...,-0.030380,0.064303,0.027336,-0.022667,0.012965,0.045342,0.028789,-0.004950,0.011055,-0.008481


# **train, test, split dan percobaan oversampling**

In [38]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
import pandas as pd

# 1. Split Data Dulu (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Inisialisasi Metode Oversampling
oversamplers = {
    "SMOTE": SMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RandomOverSampler": RandomOverSampler(random_state=42)
}

# 3. Tampung Hasil Resampling
resampled_data = {}
summary_list = []

# Hitung jumlah awal per kelas
df_awal = pd.Series(y_train).value_counts().sort_index()

for name, sampler in oversamplers.items():
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    resampled_data[name] = (X_res, y_res)

    # Hitung jumlah setelah resampling per kelas
    counts = pd.Series(y_res).value_counts().sort_index()

    summary_list.append({
        "Metode Oversampling": name,
        f"Negatif ({le.classes_[0]})": counts.get(0, 0),
        f"Netral ({le.classes_[1]})": counts.get(1, 0),
        f"Positif ({le.classes_[2]})": counts.get(2, 0),
        "Total Data Latih": len(y_res)
    })

# 4. Tampilkan Perbandingan Distribusi
df_resample_summary = pd.DataFrame(summary_list)
print("📊 DISTRIBUSI DATA TRAINING SETELAH BERBAGAI METODE OVERSAMPLING:\n")
display(df_resample_summary)

📊 DISTRIBUSI DATA TRAINING SETELAH BERBAGAI METODE OVERSAMPLING:



,Metode Oversampling,Negatif (negatif),Netral (netral),Positif (positif),Total Data Latih
0,SMOTE,2523,2523,2523,7569
1,ADASYN,2509,2523,2502,7534
2,RandomOverSampler,2523,2523,2523,7569


Teknik oversampling (SMOTE, ADASYN, ROS) berhasil menyamakan jumlah sampel data latih

# **Modeling**

In [39]:
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1. Inisialisasi 4 Algoritma Klasifikasi
base_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naïve Bayes (Gaussian)": GaussianNB(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

# Dictionary untuk menyimpan model yang sudah di-training
trained_models = {}

# 2. Loop Training 4 Model pada 3 Hasil Oversampling
print("⏳ Sedang melatih 12 kombinasi model...")

for sampler_name, (X_res, y_res) in resampled_data.items():
    for model_name, base_model in base_models.items():
        # Clone instance agar model baru dan bersih
        model = clone(base_model)

        # Fit model pada data latih hasil oversampling
        model.fit(X_res, y_res)

        # Simpan model dengan kunci gabungan (misal: "Logistic Regression + SMOTE")
        key_name = f"{model_name} + {sampler_name}"
        trained_models[key_name] = model

print("✅ Semua 12 kombinasi model berhasil di-training!")

⏳ Sedang melatih 12 kombinasi model...
✅ Semua 12 kombinasi model berhasil di-training!


# Pengecekan Overfitting vs Underfitting

In [40]:
# Cek Overfitting / Underfitting pada 12 Kombinasi Model
overfit_summary = []

for key_name, model in trained_models.items():
    # Ambil nama sampler dari kunci
    sampler_name = key_name.split(" + ")[1]
    X_res, y_res = resampled_data[sampler_name]

    # Hitung Akurasi Data Training & Testing (murni)
    train_acc = model.score(X_res, y_res)
    test_acc = model.score(X_test, y_test)
    gap = train_acc - test_acc

    overfit_summary.append({
        "Kombinasi Model & Sampler": key_name,
        "Train Acc": round(train_acc, 4),
        "Test Acc": round(test_acc, 4),
        "Gap (Train - Test)": round(gap, 4),
        "Status": "Overfitting ⚠️" if gap > 0.10 else "Good Fit ✅"
    })

# Tampilkan ke dalam DataFrame rapi
df_overfit = pd.DataFrame(overfit_summary)
display(df_overfit)

,Kombinasi Model & Sampler,Train Acc,Test Acc,Gap (Train - Test),Status
0,Decision Tree + SMOTE,1.0000,0.5874,0.4126,Overfitting ⚠️
1,Naïve Bayes (Gaussian) + SMOTE,0.6301,0.5714,0.0586,Good Fit ✅
2,Random Forest + SMOTE,1.0000,0.7660,0.2340,Overfitting ⚠️
3,Logistic Regression + SMOTE,0.8125,0.6330,0.1795,Overfitting ⚠️
4,Decision Tree + ADASYN,1.0000,0.5702,0.4298,Overfitting ⚠️
5,Naïve Bayes (Gaussian) + ADASYN,0.6346,0.5616,0.0730,Good Fit ✅
6,Random Forest + ADASYN,1.0000,0.7697,0.2303,Overfitting ⚠️
7,Logistic Regression + ADASYN,0.7953,0.6207,0.1746,Overfitting ⚠️
8,Decision Tree + RandomOverSampler,1.0000,0.6798,0.3202,Overfitting ⚠️
9,Naïve Bayes (Gaussian) + RandomOverSampler,0.6601,0.5862,0.0739,Good Fit ✅


Hasil pengecekan generalisasi menunjukkan bahwa Naïve Bayes (Gaussian) merupakan satu-satunya algoritma yang konsisten Good Fit di seluruh metode oversampling (dengan gap terendah 5.86% pada SMOTE). Sebaliknya, algoritma Decision Tree, Random Forest, dan Logistic Regression mengalami overfitting akibat kompleksitas model yang terlalu tinggi dalam mempelajari data latih ter-oversample.

Teknik oversampling (SMOTE, ADASYN, ROS) berhasil menyamakan jumlah sampel data latih, namun tidak mampu menambah variasi semantik kata pada kelas minoritas (positif). Hal ini menyebabkan model tetap mengalami trade-off antara overfitting atau kesulitan memprediksi kelas positif.

# **Evaluasi Metriks**

Fungsi spesifik masing-masing metriks:

1. Precision (Tingkat Ketepatan Tebakan):

    * Tujuan: Seberapa akurat model ketika dia menebak suatu sentimen.

    * Analogi: Dari 100 komentar yang ditedak positif oleh model, berapa yang beneran positif di dunia nyata?

2. Recall / Sensitivity (Tingkat Kepekaan / Kelengkapan Deteksi):

    * Tujuan: Seberapa baik model dalam menangkap seluruh data asli dari kelas tersebut.

    * Analogi: Dari total 96 komentar yang aslinya positif, berapa banyak yang berhasil dideteksi oleh model? (Jangan sampai banyak yang kelewatan/salah tebak jadi netral).

3. F1-Score (Keseimbangan Precision & Recall):

    * Tujuan: Angka penyeimbang (Harmonic Mean) antara Precision dan Recall. Dipakai untuk menentukan siapa pemenang model terbaik secara keseluruhan.


**INTINYA:**
Train vs Test Accuracy: Dipakai buat menjawab "Apakah model saya overfit / sehat?"

Evaluasi Metriks Detail (Precision, Recall, F1-Score): Dipakai buat menjawab "Seberapa jago dan teliti model saya dalam mengenali sentimen negatif, netral, dan positif?

In [41]:
from sklearn.metrics import classification_report

detailed_metrics = []

# Loop evaluasi untuk 12 kombinasi model yang sudah di-training
for key_name, model in trained_models.items():
    # Split nama algoritma dan sampler
    model_name, sampler_name = key_name.split(" + ")

    # Prediksi pada data testing (murni)
    y_pred = model.predict(X_test)

    # Ambil classification report dalam bentuk dictionary
    report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)

    row = {
        'Algoritma': model_name,
        'Teknik Oversampling': sampler_name
    }

    # Ambil Precision, Recall, dan F1-Score per kelas (negatif, netral, positif)
    for kelas in le.classes_:
        row[f'Precision ({kelas})'] = round(report[kelas]['precision'], 4)
        row[f'Recall ({kelas})'] = round(report[kelas]['recall'], 4)
        row[f'F1-Score ({kelas})'] = round(report[kelas]['f1-score'], 4)

    # Tambahkan Overall Accuracy & Macro Avg F1-Score
    row['Overall Accuracy'] = round(report['accuracy'], 4)
    row['Macro Avg F1-Score'] = round(report['macro avg']['f1-score'], 4)

    detailed_metrics.append(row)

# Tampilkan ke dalam DataFrame rapi, diurutkan dari Macro F1-Score tertinggi
df_metrics = pd.DataFrame(detailed_metrics).sort_values(by="Macro Avg F1-Score", ascending=False)
display(df_metrics)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/m

,Algoritma,Teknik Oversampling,Precision (negatif),Recall (negatif),F1-Score (negatif),Precision (netral),Recall (netral),F1-Score (netral),Precision (positif),Recall (positif),F1-Score (positif),Overall Accuracy,Macro Avg F1-Score
11,Logistic Regression,RandomOverSampler,0.3607,0.5605,0.4389,0.9097,0.6704,0.7719,0.0485,0.2083,0.0787,0.6355,0.4298
3,Logistic Regression,SMOTE,0.3566,0.5541,0.4339,0.9058,0.6704,0.7705,0.0396,0.1667,0.0640,0.6330,0.4228
7,Logistic Regression,ADASYN,0.3386,0.5414,0.4167,0.9022,0.6577,0.7608,0.0396,0.1667,0.0640,0.6207,0.4138
9,Naïve Bayes (Gaussian),RandomOverSampler,0.2976,0.5478,0.3857,0.8868,0.6086,0.7218,0.0667,0.2500,0.1053,0.5862,0.4042
8,Decision Tree,RandomOverSampler,0.3030,0.3185,0.3106,0.8081,0.7940,0.8010,0.0370,0.0417,0.0392,0.6798,0.3836
2,Random Forest,SMOTE,0.4342,0.2102,0.2833,0.8014,0.9334,0.8624,0.0000,0.0000,0.0000,0.7660,0.3819
6,Random Forest,ADASYN,0.4247,0.1975,0.2696,0.8038,0.9414,0.8672,0.0000,0.0000,0.0000,0.7697,0.3789
0,Decision Tree,SMOTE,0.2591,0.3631,0.3024,0.8027,0.6577,0.7230,0.0667,0.2083,0.1010,0.5874,0.3755
4,Decision Tree,ADASYN,0.2624,0.4395,0.3286,0.8211,0.6181,0.7052,0.0541,0.1667,0.0816,0.5702,0.3718
1,Naïve Bayes (Gaussian),SMOTE,0.2624,0.2357,0.2483,0.7621,0.6751,0.7160,0.0089,0.0417,0.0147,0.5714,0.3263
